# 🏴‍☠️ AN2DL Challenge 1: Pirate Pain Dataset

## 🧠 Notebook 03: Model Definition

In [1]:
import json

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from torch.nn.functional import gelu
from torch.utils.data import WeightedRandomSampler, DataLoader, Dataset

from internal.data_types import DataSetV2
from internal.persistence_manager import PersistenceManager

### Model components

The model consists of several components:
1. **TCN Building Blocks**: Dilated 1D convolutional blocks with residual connections.
2. **Attention Pooling**: An attention mechanism to pool features over the time dimension.
3. **Static MLP**: A feedforward network to process static features.
4. **PainTCNBiLSTMAttn**: The full model that integrates all components.

Additionally, we set up the loss function, optimizer, and learning rate scheduler for training.

In [2]:
# --- TCN building blocks ---

class Conv1dBlock(nn.Module):
    """
    Dilated 1D conv + GroupNorm(=LayerNorm-like) + GELU + Dropout, with residual.
    Input/Output: (B, C, T)
    """
    def __init__(self, channels: int, kernel_size: int = 5, dilation: int = 1, dropout: float = 0.3):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2  # 'same' padding
        self.conv = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm = nn.GroupNorm(1, channels)  # LayerNorm-like per-channel across (B,T)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):         # x: (B,C,T)
        residual = x
        x = self.conv(x)
        x = self.norm(x)
        x = gelu(x)
        x = self.drop(x)
        return x + residual       # residual

class TCN(nn.Module):
    """Stack of dilated residual Convolution 1d blocks."""
    def __init__(self, channels: int, dilations=(1,2,4,8), kernel_size: int = 5, dropout: float = 0.3):
        super().__init__()
        self.blocks = nn.ModuleList([
            Conv1dBlock(channels, kernel_size=kernel_size, dilation=d, dropout=dropout) for d in dilations
        ])
    def forward(self, x):         # (B,C,T)
        for b in self.blocks:
            x = b(x)
        return x

# --- Attention pooling over time ---

class AttnPool(nn.Module):
    """
    Additive attention over time.
    Input: (B, T, C)
    Output: (B, C)
    """
    def __init__(self, in_dim: int, hidden: int = 128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):         # x: (B,T,C)
        a = self.score(x).squeeze(-1)         # (B,T)
        a = torch.softmax(a, dim=1)           # (B,T)
        return torch.sum(x * a.unsqueeze(-1), dim=1)  # (B,C)

# --- Static MLP ---

class StaticMLP(nn.Module):
    def __init__(self, in_dim=7, hidden=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout)
        )
    def forward(self, x):         # (B,7)
        return self.net(x)        # (B,64)

# --- Full model ---

class PainTCNBiLSTMAttn(nn.Module):
    """
    Inputs:
      x_num:  (B, T=160, D_num)    # engineered numeric dynamics: 90 (30 raw + 30 delta + 30 rollstd)
      x_surv: list of 4 (B, T)     # each int in {0,1,2} for pain_survey_1..4
      x_sta:  (B, 7)               # static features (scaled joint_30 + counts + binaries)
    """
    def __init__(self, d_num: int = 90, d_emb: int = 2, tcn_channels: int = 128,
                 lstm_hidden: int = 128, num_classes: int = 3, dropout: float = 0.3):
        super().__init__()

        # 4 survey embeddings (3 categories each)
        self.survey_embs = nn.ModuleList([nn.Embedding(3, d_emb) for _ in range(4)])

        # Project timestep features to TCN channels
        d_in = d_num + 4 * d_emb
        self.proj = nn.Linear(d_in, tcn_channels)

        # TCN front
        self.tcn = TCN(channels=tcn_channels, dilations=(1,2,4,8), kernel_size=5, dropout=dropout)

        # BiLSTM over (B,T,C)
        self.lstm = nn.LSTM(input_size=tcn_channels, hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True, bidirectional=True, dropout=0.0)

        # Attention pooling over time
        self.attn = AttnPool(in_dim=2*lstm_hidden, hidden=128)

        # Static branch
        self.static_mlp = StaticMLP(in_dim=7, hidden=64, dropout=dropout)

        # Fusion head
        self.head = nn.Sequential(
            nn.Linear(2*lstm_hidden + 64, 128),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, _x_num, x_surv_list, _x_sta):
        """
        x_num: (B,T,D_num)
        x_surv_list: [s1,s2,s3,s4] each (B,T) long
        x_sta: (B,7)
        """
        B, T, _ = _x_num.shape
        # Embeddings for the 4 surveys
        embs = [emb(surv) for emb, surv in zip(self.survey_embs, x_surv_list)]  # 4 × (B,T,d_emb)
        x = torch.cat([_x_num, *embs], dim=-1)                                   # (B,T, d_num+4*d_emb)

        # Project to TCN channels
        x = self.proj(x)                       # (B,T,C)
        # TCN expects (B,C,T)
        x = x.transpose(1, 2)                  # (B,C,T)
        x = self.tcn(x)                        # (B,C,T)
        x = x.transpose(1, 2)                  # (B,T,C)

        # BiLSTM
        x, _ = self.lstm(x)                    # (B,T,2*hidden)

        # Attention pooling
        dyn_vec = self.attn(x)                 # (B,2*hidden)

        # Static branch
        sta_vec = self.static_mlp(_x_sta)       # (B,64)

        # Fusion
        z = torch.cat([dyn_vec, sta_vec], dim=-1)   # (B, 2*hidden + 64)
        _logits = self.head(z)                       # (B,3)
        return _logits


In [3]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.05, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.register_buffer('weight', weight if weight is not None else None)

    def forward(self, _logits, target):
        n_class = _logits.size(-1)
        log_prob = torch.log_softmax(_logits, dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_prob)
            true_dist.fill_(self.smoothing / (n_class - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        # class weights applied as expected value over classes
        if self.weight is not None:
            w = self.weight.unsqueeze(0)  # (1,C)
            _loss = (-true_dist * log_prob * w).sum(dim=-1)
        else:
            _loss = (-true_dist * log_prob).sum(dim=-1)
        return _loss.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.register_buffer('alpha', alpha if alpha is not None else None)

    def forward(self, _logits, target):
        logpt = torch.log_softmax(_logits, dim=-1)           # (B,C)
        pt    = torch.exp(logpt)                             # (B,C)
        # gather log-prob and prob of the true class
        logpt_t = logpt.gather(1, target.unsqueeze(1)).squeeze(1)  # (B,)
        pt_t    = pt.gather(1, target.unsqueeze(1)).squeeze(1)     # (B,)
        _loss = - (1 - pt_t) ** self.gamma * logpt_t                # (B,)
        if self.alpha is not None:
            alpha_t = self.alpha[target]                            # (B,)
            _loss = alpha_t * _loss
        if self.reduction == 'mean':
            return _loss.mean()
        elif self.reduction == 'sum':
            return _loss.sum()
        else:
            return _loss

def normalize_class_weights(w):
    w = w / w.sum()
    return w

def make_weighted_sampler(y_train_fold, _class_weights):
    # map each sample to its class weight
    weights = _class_weights.cpu().numpy()
    sample_w = np.take(weights, y_train_fold)  # (N,)
    return WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)


In [4]:
class PainDataset(Dataset):
    def __init__(self, _x_dyn_num, _x_surv, _x_sta, _y=None):
        self.X_dyn_num = torch.from_numpy(_x_dyn_num).float()
        self.X_sta     = torch.from_numpy(_x_sta).float()
        self.X_surv    = [torch.from_numpy(s).long() for s in _x_surv] if _x_surv is not None else None
        self.y         = torch.from_numpy(_y).long() if _y is not None else None

    def __len__(self):
        return self.X_dyn_num.shape[0]

    def __getitem__(self, idx):
        item = {
            'x_num':  self.X_dyn_num[idx],              # (160, 90)
            'x_surv': [s[idx] for s in self.X_surv],    # list of 4 (160,)
            'x_sta':  self.X_sta[idx],                  # (7,)
        }
        if self.y is not None:
            item['y'] = self.y[idx]
        return item


In [5]:
# Robust loading: try v2, otherwise fall back to the older loader.
# Remove the variable annotation to avoid accidental instantiation issues.
try:
    data = PersistenceManager.load_arrays_v2()
except TypeError as e_v2:
    try:
        data = PersistenceManager.load_arrays()
        print("Fallback: loaded data with load_arrays() due to:", e_v2)
    except Exception as e2:
        raise RuntimeError("Could not load dataset via load_arrays_v2() or load_arrays().") from e2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PainTCNBiLSTMAttn(d_num=90, d_emb=2, tcn_channels=128, lstm_hidden=128, num_classes=3, dropout=0.3).to(device)

train_dataset = PainDataset(
    data.X_dyn_num_train,
    data.X_surv_train,
    data.X_sta_train,
    data.y_train
)
val_dataset   = PainDataset(
    data.X_dyn_num_val,
    data.X_surv_val,
    data.X_sta_val,
    data.y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,          # randomize samples each epoch
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# Example batch shapes:
# x_num:  (B,160,90)      -> float32
# x_surv: [ (B,160), ...] -> int64
# x_sta:  (B,7)           -> float32
# y:      (B,)            -> int64 in {0,1,2}

def make_class_weights(y_train_fold, _device):
    classes = np.array([0,1,2])
    w = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_fold)
    w = torch.tensor(w, dtype=torch.float32, device=_device)
    return w

# Weighted CE
class_weights = make_class_weights(data.y_train, device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
# TODO: Sometimes stabilizes with imbalance, use instead of plain CE:
# criterion = LabelSmoothingCE(smoothing=0.05, weight=class_weights)
# TODO: Use if high_pain recall < 0.55 after a few epochs:
#       don't combine Focal + WeightedRandomSampler at the same time initially. Try Focal or Sampler.
# alpha = normalize_class_weights(class_weights)     # tensor on device
# criterion = FocalLoss(alpha=alpha, gamma=2.0)
# TODO: Turn this on only if CE (or LS-CE) still under-recalls high_pain.
#       If you use the sampler, keep the loss as plain CE or LabelSmoothingCE (not focal), to avoid over-focusing instability.
# sampler = make_weighted_sampler(data.y_train, class_weights)
# train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4, pin_memory=True)

# Optimizer / schedule
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=2e-5)


Arrays v2 loaded successfully from: /home/andre/university/AN2DL-Challenge-1/notebooks/processed/arrays_v2.joblib


In [6]:
@torch.no_grad()
def evaluate_metrics(_model, _loader, _device):
    _model.eval()
    y_true, y_pred = [], []
    for _batch in _loader:
        _x_num  = _batch['x_num'].to(_device)         # (B,160,90)
        _x_sta  = _batch['x_sta'].to(_device)         # (B,7)
        _x_surv = [s.to(_device) for s in _batch['x_surv']]  # list of 4 (B,160)
        _y      = _batch['y'].to(_device)             # (B,)
        _logits = _model(_x_num, _x_surv, _x_sta)       # (B,3)
        y_hat  = _logits.argmax(dim=1)
        y_true.append(_y.cpu().numpy())
        y_pred.append(y_hat.cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    _macro_f1  = f1_score(y_true, y_pred, average='macro', labels=[0,1,2])
    _rec_pc    = recall_score(y_true, y_pred, average=None, labels=[0,1,2])  # per-class recall
    _cm        = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    return _macro_f1, _rec_pc, _cm


In [ ]:
best_f1, best_state = -1.0, None
bad = 0
EPOCHS, PATIENCE = 80, 12

for epoch in range(1, EPOCHS+1):
    model.train()
    running = 0.0
    for batch in train_loader:
        x_num  = batch['x_num'].to(device)
        x_sta  = batch['x_sta'].to(device)
        x_surv = [s.to(device) for s in batch['x_surv']]
        y      = batch['y'].to(device)

        optimizer.zero_grad()
        logits = model(x_num, x_surv, x_sta)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item() * y.size(0)

    train_loss = running / len(train_loader.dataset)

    # --- validation ---
    macro_f1, rec_pc, cm = evaluate_metrics(model, val_loader, device)
    scheduler.step(macro_f1)   # ReduceLROnPlateau on macro-F1 (maximize)

    print(f"[{epoch:02d}] loss={train_loss:.4f} | F1(macro)={macro_f1:.4f} | "
          f"recalls={rec_pc} | lr={optimizer.param_groups[0]['lr']:.2e}")

    # early stopping on macro-F1
    if macro_f1 > best_f1:
        best_f1, bad = macro_f1, 0
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE:
            print("Early stopping.")
            break

# restore best
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
final_f1, final_rec, final_cm = evaluate_metrics(model, val_loader, device)
print("BEST macroF:", round(final_f1,4), "recalls:", np.round(final_rec,3))
torch.save(best_state, 'fold0_best.pt')
try:
    json.dump({'macroF':final_f1, 'recall':final_rec.tolist(), 'cm':final_cm.tolist()}, open('fold0_metrics.json','w'), indent=2)
except Exception as e:
    print("Could not save metrics json:", e)
print("Saved:", 'fold0_best.pt')

[01] loss=1.0865 | F1(macro)=0.4395 | recalls=[0.74757282 0.16666667 0.58333333] | lr=1.00e-03
[02] loss=0.9492 | F1(macro)=0.5678 | recalls=[0.81553398 0.44444444 0.58333333] | lr=1.00e-03
[03] loss=0.7059 | F1(macro)=0.6100 | recalls=[0.85436893 0.66666667 0.41666667] | lr=1.00e-03
[04] loss=0.6203 | F1(macro)=0.6675 | recalls=[0.89320388 0.61111111 0.58333333] | lr=1.00e-03
[05] loss=0.4529 | F1(macro)=0.7020 | recalls=[0.87378641 0.72222222 0.66666667] | lr=1.00e-03
[06] loss=0.4585 | F1(macro)=0.6575 | recalls=[0.85436893 0.55555556 0.75      ] | lr=1.00e-03
[07] loss=0.2871 | F1(macro)=0.7590 | recalls=[0.90291262 0.83333333 0.66666667] | lr=1.00e-03
[08] loss=0.2509 | F1(macro)=0.6840 | recalls=[0.93203883 0.38888889 0.83333333] | lr=1.00e-03
[09] loss=0.2839 | F1(macro)=0.6718 | recalls=[0.80582524 0.88888889 0.58333333] | lr=1.00e-03
[10] loss=0.1949 | F1(macro)=0.7207 | recalls=[0.84466019 0.66666667 0.91666667] | lr=1.00e-03
[11] loss=0.2151 | F1(macro)=0.7884 | recalls=[0.9

In [8]:
final_f1

0.8675306978832783

In [9]:
final_rec

array([0.98058252, 0.94444444, 0.66666667])

In [10]:
final_cm

array([[101,   1,   1],
       [  0,  17,   1],
       [  2,   2,   8]])